In [0]:
dbutils.widgets.text("process_date", "")
process_date = dbutils.widgets.get("process_date")

In [0]:
from pyspark.sql.functions import col, lit

df_licitaciones = (
    spark.table("chilecompra.bronze.licitaciones_api")
    .filter(
        col("_process_date") == lit(process_date).cast("date")
    )
)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

w = (
    Window
    .partitionBy("codigo_externo")
    .orderBy(
        col("_process_date").desc(),
        col("_source_created_at").desc(),
        col("fecha_cierre").desc(),
        col("_ingestion_timestamp").desc()
    )
)

df_licitaciones_latest = (
    df_licitaciones
    .withColumn("_rn", row_number().over(w))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

In [0]:
df_licitaciones_merge = (
    df_licitaciones_latest
    .select(
        col("codigo_externo"),
        col("nombre"),
        col("codigo_estado"),
        col("fecha_cierre"),

        col("_process_date").alias("_last_api_process_date"),
        col("_source_created_at").alias("_last_api_source_created_at"),
        col("_source_version").alias("_last_api_version")
    )
)

In [0]:
total_rows = df_licitaciones_merge.count()

null_codigo = (
    df_licitaciones_merge
    .filter(col("codigo_externo").isNull())
    .count()
)

distinct_codigo = (
    df_licitaciones_merge
    .select("codigo_externo")
    .distinct()
    .count()
)

duplicate_codigo = total_rows - distinct_codigo

if null_codigo > 0:
    raise ValueError(
        f"DQ FAILED: {null_codigo} licitaciones have codigo_externo NULL"
    )

if duplicate_codigo > 0:
    raise ValueError(
        f"DQ FAILED: {duplicate_codigo} duplicated codigo_externo "
        f"after latest selection"
    )

print(
    f"Pre-merge licitaciones DQ passed: "
    f"{total_rows} licitaciones"
)

In [0]:
from delta.tables import DeltaTable

target_table = "chilecompra.silver.licitaciones"

api_is_newer = """
    t._last_api_process_date IS NULL

    OR s._last_api_process_date > t._last_api_process_date

    OR (
        s._last_api_process_date = t._last_api_process_date
        AND (
            t._last_api_source_created_at IS NULL
            OR s._last_api_source_created_at > t._last_api_source_created_at
        )
    )
"""

In [0]:
if not spark.catalog.tableExists(target_table):

    (
        df_licitaciones_merge.write
        .format("delta")
        .saveAsTable(target_table)
    )

else:

    delta_target = DeltaTable.forName(
        spark,
        target_table
    )

    (
        delta_target.alias("t")
        .merge(
            df_licitaciones_merge.alias("s"),
            "t.codigo_externo = s.codigo_externo"
        )
        .whenMatchedUpdate(
            condition=api_is_newer,
            set={
                "nombre": "s.nombre",
                "codigo_estado": "s.codigo_estado",
                "fecha_cierre": "s.fecha_cierre",
                "_last_api_process_date": "s._last_api_process_date",
                "_last_api_source_created_at": "s._last_api_source_created_at",
                "_last_api_version": "s._last_api_version"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
df_silver_licitaciones = spark.table(target_table)

silver_rows = df_silver_licitaciones.count()

silver_distinct = (
    df_silver_licitaciones
    .select("codigo_externo")
    .distinct()
    .count()
)

silver_null_codigo = (
    df_silver_licitaciones
    .filter(col("codigo_externo").isNull())
    .count()
)

missing_in_silver = (
    df_licitaciones_merge
    .select("codigo_externo")
    .join(
        df_silver_licitaciones.select("codigo_externo"),
        on="codigo_externo",
        how="left_anti"
    )
    .count()
)

if missing_in_silver > 0:
    raise ValueError(
        f"DQ FAILED: {missing_in_silver} current licitaciones "
        f"are missing from Silver"
    )

if silver_rows != silver_distinct:
    raise ValueError(
        f"DQ FAILED: rows={silver_rows}, "
        f"distinct codigo_externo={silver_distinct}"
    )

if silver_null_codigo > 0:
    raise ValueError(
        f"DQ FAILED: {silver_null_codigo} rows have codigo_externo NULL"
    )

print(
    f"Silver licitaciones DQ passed: "
    f"{silver_rows} licitaciones, "
    f"all current source licitaciones present"
)

In [0]:
invalid_versions = (
    df_licitaciones_merge.alias("a")
    .join(
        df_silver_licitaciones.alias("s"),
        on="codigo_externo",
        how="inner"
    )
    .filter(
        (
            col("s._last_api_process_date") < col("a._last_api_process_date")
        )
        |
        (
            (col("s._last_api_process_date") == col("a._last_api_process_date"))
            &
            (
                col("s._last_api_source_created_at")
                < col("a._last_api_source_created_at")
            )
        )
    )
    .count()
)

if invalid_versions > 0:
    raise ValueError(
        f"DQ FAILED: {invalid_versions} licitaciones in Silver "
        f"are older than the current API version"
    )

print("Licitaciones freshness DQ passed")